# 2 — Training: the **native point head** on CFD Brackish

**What is being tested.** The parked branch trained a box head (heat + `wh` + `off`) on these frames; rescored as points it reached **F1 0.758** (best) / 0.746 (last) against released **RF-DETR-Nano at 0.782**. This notebook trains the same decoder with **only** the heat branch — `crop-counter`'s native task — on the same frames, same recipe, same seed. Two readings are possible and the run distinguishes them: the geometry branch was *regularising* the shared trunk (dropping it costs the heat branch), or it was *competing* for a 3.4 M-parameter budget (dropping it frees capacity).

**One seed. This is a read, not a claim.** A single run cannot separate a real effect from seed noise; it can say whether the point head lands in the same neighbourhood, far above it, or far below it. Anything tighter than "neighbourhood" needs seeds we are not paying for yet.

**A real difference in the data diet, named before the numbers.** The box run's sampler had `negative_tile_fraction = 0.2`: 80 % of tiles were re-cropped until a box survived, so it saw ≈**9.2k positive tiles per epoch**. This point path has no such knob — `CropTileDataset` samples all 11,547 frames uniformly at 1 tile/frame, and ~60 % of Brackish frames are empty, so it sees ≈**4.6k positive tiles per epoch**, roughly half. The penalty-reduced focal loss clamps `n_pos ≥ 1`, so an all-empty batch is not a numerical hazard — but half the positive exposure per epoch is a genuine confound between the two runs, not a footnote. It goes in the report (§ 4) beside any comparison, in both directions: a point head that matches the box head did so on half the positive tiles; one that loses may be losing to the diet, not the head shape.

**Order of operations.** Backbone verified by reproducing a committed metric → the two silent config traps asserted → the tiles looked at → then, and only then, 8 epochs of A100 time.

> Run **`1_reformat.ipynb` first**, or let § 1b below rebuild the slice on this runtime (~5 min).

## 1 · Drive, paths, code

Mounts Drive, fixes the paths every cell below uses, clones this branch and installs it editable. Safe to re-run: the clone is wiped and redone each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, shutil, importlib, pathlib
from pathlib import Path

from IPython.display import Image as IPImage, display

DRIVE  = '/content/drive/MyDrive/frozen-trunk-detection'
REPO   = '/content/crop-counter'
DATA   = '/content/data/brackish'          # the COCO *bbox* subset — the pixels live here
POINTS = '/content/data/brackish_points'   # the COCO *keypoints* root the trainer reads
CFD    = '/content/cfd'
RUN    = f'{DRIVE}/runs/brackish_points_s0'
RES    = f'{DRIVE}/results/points'
for d in (CFD, f'{DRIVE}/weights', f'{DRIVE}/runs', f'{DRIVE}/results', RES, f'{RES}/figures'):
    os.makedirs(d, exist_ok=True)
print(os.listdir(DRIVE))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/fish-points --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# cfd = ijson (streams the 1.9M-record CFD metadata); dev = nbconvert/ruff/pytest;
# portal = the hosted-API client. No detection extra on this branch: no boxes, no COCO AP.
!pip install -q -e ".[dev,portal,cfd]"

# A running kernel never re-reads site-packages' .pth files, so the editable install is
# invisible to THIS process until a restart (subprocess `!python -m ...` calls see it fine).
for p in ('/content/crop-counter/src', '/content/crop-counter/examples/FishDetection/scripts'):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
import cropcounter, torch
import nb_helpers as nbh
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())

# Backbone: Meta's DINOv3 checkpoint is gated; the file on Drive is the ungated timm
# re-host converted to Meta's parameter names. Symlink, never copy — it is 350 MB and the
# clone is thrown away every session anyway.
os.makedirs('weights', exist_ok=True)
BACKBONE = 'dinov3_convnext_base_pretrain_lvd1689m-801f2ba9.pth'
if not os.path.lexists(f'weights/{BACKBONE}'):
    os.symlink(f'{DRIVE}/weights/{BACKBONE}', f'weights/{BACKBONE}')
# decoder_best.pt (the shipped wheat decoder) is what § 3's gate reproduces. It is
# tracked in git; a Drive copy, if one exists, wins.
if os.path.exists(f'{DRIVE}/weights/decoder_best.pt'):
    if os.path.lexists('weights/decoder_best.pt'):
        os.remove('weights/decoder_best.pt')
    os.symlink(f'{DRIVE}/weights/decoder_best.pt', 'weights/decoder_best.pt')
    print('decoder_best.pt <- Drive')
else:
    print("decoder_best.pt: using the repo's own (tracked in git)")
!ls -l weights/

## 1b · Make sure the Brackish point slice is on this VM

Colab gives each notebook its own runtime, so the frames `1_reformat.ipynb` fetched are not here unless this notebook is attached to that same session. Idempotent: it rebuilds only what is missing (metadata 47 MB, 14,674 frames off the LILA GCS mirror, ~4–5 min).

In [ ]:
def _n_files(path):
    return len(os.listdir(path)) if os.path.isdir(path) else 0

need_data = not (os.path.exists(f'{DATA}/val/annotations.json')
                 and _n_files(f'{DATA}/val/images') > 0)
need_points = not os.path.exists(f'{POINTS}/val/annotations.json')

if need_data or need_points:
    META = f'{CFD}/community_fish_detection_dataset.json.zip'
    if not os.path.exists(META):
        !wget -q -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
if need_data:
    !python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources brackish_dataset --train-cap 100000 --val-cap 100000 --seed 0 --no-progress
    !python -m cropcounter.cfd fetch --subset {DATA} --max-side 1024 --workers 32 --mirror gcs --no-progress
if need_points:
    !python -m cropcounter.cfd points --subset {DATA} --out {POINTS}

for split in ('train', 'val'):
    print(f'{split}: {_n_files(f"{DATA}/{split}/images"):,} frames | '
          f'points root {"ok" if os.path.exists(f"{POINTS}/{split}/annotations.json") else "MISSING"}')
print(json.dumps(json.load(open(f'{POINTS}/points_summary.json')), indent=1))

## 2 · Machine

**This branch hard-codes bf16 autocast on CUDA** — `train.py`, `metrics._iter_prob_maps` and `inference.predict_prob` all pass `dtype=torch.bfloat16` with no GradScaler and no fp16 fallback. A T4 is sm_75: it has no bf16 hardware path, so the run would either crash or silently fall back to an emulated path that is slower than fp32 and numerically unlike the numbers reported here. The assertion below is therefore a hard gate, not a preference: **run this on an Ampere+ GPU (A100/L4), or change the package, but do not run it on a T4 and compare the result to anything.**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('vCPUs:', os.cpu_count())
assert torch.cuda.is_available(), 'no GPU — Runtime > Change runtime type > A100'
print('GPU:', torch.cuda.get_device_name(0),
      '| compute capability:', torch.cuda.get_device_capability())
assert torch.cuda.get_device_capability()[0] >= 8, \
    'this branch hard-codes bf16 autocast; run on Ampere+ (A100)'
!df -h /content | tail -1

## 3 · Backbone weights + metric-reproduction gate

The DINOv3 checkpoint is Meta-gated; the file on Drive is the ungated **timm re-host converted to Meta's parameter names** (vault memory `reference_dinov3_gated_weights`). A strict `load_checkpoint` only proves the *shapes* line up — a plausibly-wrong tensor set loads just as cleanly as the right one. So the gate verifies by **reproducing a metric**: the shipped wheat decoder `weights/decoder_best.pt` (tracked in git) run over the six committed example val images must reproduce `examples/demo/data/val/expected_counts.csv`.

Those reference counts were produced **on MPS in fp32**, so the asserted comparison runs fp32 with **TF32 off** — like for like. CUDA fp32 is not bit-identical to MPS fp32 (different cuDNN algorithms) and Ampere runs fp32 convolutions in TF32 by default, so the tolerance is `max(1, 5 %)` per image. TF32 and bf16 (what training actually uses) are run too and merely **printed**: on a dense frame a couple of threshold-straddling peaks move, which is a precision difference, not a backbone difference. A wrong backbone wrecks all six.

In [ ]:
import csv

from torch.utils.data import DataLoader

from cropcounter.crop_dataset import CropTileDataset, collate_val, load_records
from cropcounter.metrics import evaluate
from cropcounter.train import load_checkpoint

device = torch.device('cuda')
gate_model, gate_cfg = load_checkpoint(Path('weights/decoder_best.pt'), device,
                                       weights_dir=Path('weights'))
gate_model.eval()
print(f'wheat decoder loaded strictly | backbone {gate_cfg.backbone} | labels {gate_cfg.labels} '
      f'| stride {gate_cfg.output_stride} | k {gate_cfg.k} | nms {gate_cfg.nms_radius}')

gate_recs = load_records(Path('examples/demo/data/val'), fmt='cvat')
gate_ds = CropTileDataset(gate_recs, Path('examples/demo/data/val/images'), train=False,
                          output_stride=gate_cfg.output_stride, sigma=gate_cfg.sigma)
expected = {r['image_name']: int(r['predicted_count'])
            for r in csv.DictReader(open('examples/demo/data/val/expected_counts.csv'))}

GATE_TAU = 0.35   # the tau the committed counts were swept to (weights/README.md)

def gate_count(image, record, amp, tf32):
    defaults = (torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32)
    torch.backends.cudnn.allow_tf32 = tf32
    torch.backends.cuda.matmul.allow_tf32 = tf32
    try:
        points, _, _ = nbh.decode_image_points(
            gate_model, image, record.width, record.height, device, tau=GATE_TAU,
            k=gate_cfg.k, nms_radius=gate_cfg.nms_radius,
            output_stride=gate_cfg.output_stride, amp=amp)
    finally:
        torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32 = defaults
    return len(points)

ok = True
print(f'{"image":10s} {"fp32/noTF32":>12s} {"committed":>10s} {"D":>4s}  ||  '
      f'{"fp32/TF32":>10s} {"bf16":>6s}')
for i, record in enumerate(gate_recs):
    image = gate_ds[i]['image']
    c_fp32 = gate_count(image, record, amp=False, tf32=False)
    c_tf32 = gate_count(image, record, amp=False, tf32=True)
    c_bf16 = gate_count(image, record, amp=True, tf32=True)
    exp = expected[record.name]
    ok &= abs(c_fp32 - exp) <= max(1, round(0.05 * exp))
    print(f'{record.name:10s} {c_fp32:12d} {exp:10d} {c_fp32 - exp:+4d}  ||  '
          f'{c_tf32:10d} {c_bf16:6d}')

gate_loader = DataLoader(gate_ds, batch_size=1, collate_fn=collate_val)
summary, _ = evaluate(gate_model, gate_loader, device, tau=GATE_TAU, k=gate_cfg.k,
                      nms_radius=gate_cfg.nms_radius, output_stride=gate_cfg.output_stride,
                      match_radius_px=gate_cfg.match_radius_px)
print({k: round(v, 3) for k, v in summary.items()})
assert ok, ('the converted backbone does NOT reproduce the committed wheat counts '
            '(fp32, TF32 off, 5 % gate) — stop here, everything downstream is meaningless')
print('backbone verified by fp32 metric reproduction | training will run under bf16')

del gate_model, gate_ds, gate_loader

## 4 · The config, the splits, and the two silent traps

[`config_points_8ep.json`](../config_points_8ep.json) mirrors the box run's recipe minus every box-only field: frozen `base` backbone, `c_dec` 192, stride 4, `sigma` 2.0, 768-px tiles at **1 tile/frame**, batch 8, 8 epochs, lr 1e-3 with 2 warm-up epochs, seed 0. Its `data_root` is the keypoints root and its `out_dir` already points at Drive, so checkpoints are written straight there as they are produced.

Two config defaults are wrong for fish and **both fail silently**:

- `labels` defaults to `("Wheat", "Volunteer")`. Every `fish` point would be filtered out and the run would train on an empty dataset with a finite, falling loss.
- `augment_profile` defaults to `"wheat"`, which adds `VerticalFlip` + `RandomRotate90` — label-preserving for nadir crop photos, wrong for underwater footage, which has an up.

The assertion below is the guard for the first: the loaded point count must equal the **12,155** boxes in the bbox subset. The second is asserted in § 5 by reading the transform back. (`annotation_format` is the third trap and is the loud one — it defaults to `"cvat"` and raises on a missing `annotations.xml`.)

In [ ]:
from cropcounter.crop_dataset import load_splits
from cropcounter.train import TrainConfig

CONFIG = 'examples/FishDetection/config_points_8ep.json'
cfg = TrainConfig.from_json(Path(CONFIG))
print(json.dumps(cfg.to_dict(), indent=1))
assert str(cfg.data_root) == POINTS, f'config data_root {cfg.data_root} is not {POINTS}'
assert str(cfg.out_dir) == f'{DRIVE}/runs' and cfg.run_name == 'brackish_points_s0'
assert cfg.annotation_format == 'coco' and cfg.augment_profile == 'natural'

train_recs, val_recs = load_splits(cfg.data_root, fmt=cfg.annotation_format, labels=cfg.labels)
n_train_points = sum(len(r.points) for r in train_recs)
n_val_points = sum(len(r.points) for r in val_recs)
n_train_empty = sum(1 for r in train_recs if not r.points)

# The COUNTED_LABELS trap: with the default labels this is 0 and training still "works".
assert n_train_points == 12155, (
    f'{n_train_points} train points, expected 12,155 — labels={cfg.labels} is filtering '
    'fish out (COUNTED_LABELS defaults to the wheat labels)')
print(f'train: {len(train_recs):,} images | {n_train_points:,} points (== 12,155 ) | '
      f'{n_train_empty:,} empty ({100 * n_train_empty / len(train_recs):.1f}%)')
print(f'val:   {len(val_recs):,} images | {n_val_points:,} points')
assert n_val_points == 1965, n_val_points

## 5 · What the head trains on

Eight augmented training tiles exactly as the model receives them, with the rendered peak-1.0 Gaussian target on top. Three things to look for:

1. **No upside-down fish.** `natural` drops `VerticalFlip` and `RandomRotate90`; the assertions below read the transform back rather than trusting the config, and the panel is the visual confirmation.
2. **Every hot blob sits on an animal.** A blob in open water is a target-geometry bug (`sigma` / stride / keypoint scaling), and it is far cheaper to see it here than to infer it from a flat loss curve.
3. **Black bars.** Brackish frames are 960×540 and the tile is 768, so `RandomCrop(pad_if_needed=True)` constant-pads the short axis. That is the same padding the box run used — real, not a bug, and it is part of why 1 tile/frame is the right setting here.

In [ ]:
transform_repr = repr(CropTileDataset.default_transform(
    cfg.tile, cfg.scale_jitter, profile=cfg.augment_profile))
print(transform_repr)
assert 'VerticalFlip' not in transform_repr, \
    f'augment_profile={cfg.augment_profile!r} still flips vertically — underwater has an up'
assert 'RandomRotate90' not in transform_repr, \
    f'augment_profile={cfg.augment_profile!r} still rotates by 90 degrees'
assert 'HorizontalFlip' in transform_repr, 'the natural profile should keep horizontal flips'
print('\naugmentation is the natural profile: no VerticalFlip, no RandomRotate90 ')

tile_ds = CropTileDataset(
    train_recs, cfg.train_images_dir, train=True, tile=cfg.tile,
    output_stride=cfg.output_stride, sigma=cfg.sigma, tiles_per_image=cfg.tiles_per_image,
    scale_jitter=cfg.scale_jitter, augment_profile=cfg.augment_profile,
)
path = nbh.plot_train_tiles_with_targets(
    tile_ds, f'{RES}/figures/train_tiles_targets.png', n=8, seed=0,
    output_stride=cfg.output_stride,
    title=(f"What the point head trains on — {cfg.tile}px tiles, '{cfg.augment_profile}' "
           f"augmentation, stride-{cfg.output_stride} peak-1.0 Gaussian targets (hot). "
           "A blob off its fish, or an upside-down fish, is a bug."),
)
display(IPImage(str(path)))
del tile_ds

## 6 · Train — 8 epochs, seed 0

The trainer runs as a **subprocess**, not in-kernel: it gets a clean CUDA context, and a dead kernel does not take the run with it. `{RUN}` is created first because `tee` opens the log before `train.py` creates the run directory.

The cell above it frees the GPU the notebook itself is holding (the backbone gate built a whole model). If it still reports more than ~2 GB reserved, restart the runtime and run from the top — the VM disk survives a restart, so nothing is refetched.

In [ ]:
import gc

for name in ('gate_model', 'gate_ds', 'gate_loader', 'tile_ds', 'model'):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
reserved_gb = torch.cuda.memory_reserved() / 1e9
print(f'kernel reserved {reserved_gb:.2f} GB before launching the trainer')
assert reserved_gb < 2.0, 'the kernel is holding the GPU — Runtime > Restart session, then Run all'
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
os.makedirs(RUN, exist_ok=True)

In [ ]:
t0 = time.time()
!python -m cropcounter.train --config {CONFIG} 2>&1 | tee {RUN}/train.log
print(f'\ntrained in {(time.time() - t0) / 60:.1f} min')
!ls -l {RUN}

## 7 · Curves and history

`best.pt` is selected by **lowest validation loss** — the penalty-reduced focal loss against the heatmap targets, which is τ-independent. The per-epoch P/R/F1 and MAE printed here are decoded at the config's `tau` 0.3, a training-time default and *not* a calibrated threshold; § 8 calibrates it properly. Read the two columns separately: a val loss that bottoms early while the detection metrics keep climbing is exactly what the box run did, and § 10 argues what to do about it.

In [ ]:
history = json.load(open(f'{RUN}/history.json'))
shutil.copy(f'{RUN}/curves.png', f'{RES}/figures/curves.png')

val_loss = [float('inf') if v != v else v for v in history['val_loss']]
best_epoch = int(min(range(len(val_loss)), key=lambda i: val_loss[i])) + 1
n_epochs = len(history['train_loss'])

print(f'{"ep":>3} {"train":>8} {"val":>8} {"MAE":>7} {"RMSE":>7} {"bias":>7} '
      f'{"P":>6} {"R":>6} {"F1":>6} {"lr":>9}')
for i in range(n_epochs):
    mark = '  <- best (lowest val loss)' if i + 1 == best_epoch else ''
    print(f'{i + 1:3d} {history["train_loss"][i]:8.4f} {history["val_loss"][i]:8.4f} '
          f'{history["val_count_mae"][i]:7.2f} {history["val_count_rmse"][i]:7.2f} '
          f'{history["val_count_bias"][i]:+7.2f} {history["val_precision"][i]:6.3f} '
          f'{history["val_recall"][i]:6.3f} {history["val_f1"][i]:6.3f} '
          f'{history["lr"][i]:9.2e}{mark}')
print(f'\nbest epoch (lowest val loss): {best_epoch} of {n_epochs} | '
      f'last epoch F1 at tau 0.3: {history["val_f1"][-1]:.3f} '
      f'vs best-epoch {history["val_f1"][best_epoch - 1]:.3f}')
display(IPImage(f'{RES}/figures/curves.png'))

## 8 · τ calibration

`tau` is a **decode-time** threshold, not a trained parameter: the same weights give different numbers at different τ, so the only honest comparison is swept against swept. `metrics.sweep_tau` computes each val image's probability map **once** and decodes it at every τ, so a 14-point sweep costs one forward pass over the 3,127 val frames.

Four sweeps: `best.pt` and `last.pt`, each at `nms_radius` **1.5** (this experiment's default, and the separation floor implied by `k=3`) and **5** (the wheat report's choice, where smeared blobs were decoding as duplicate points). τ from 0.05 to 0.70 in steps of 0.05.

**Selection rule, fixed before the numbers:** τ is the **argmin of count MAE**, following the wheat report. The F1 argmax is computed and printed as a sanity check — if the two disagree materially, that disagreement is the finding (a τ that counts well but localises badly means compensating errors), and it is reported rather than resolved by picking the flattering one.

In [ ]:
import numpy as np

val_ds = CropTileDataset(val_recs, cfg.val_images_dir, train=False,
                         output_stride=cfg.output_stride, sigma=cfg.sigma)
TAUS = np.round(np.arange(0.05, 0.75, 0.05), 2).tolist()
NMS_RADII = (1.5, 5.0)
print('taus:', TAUS)

calibrations = {}
models = {}
for ckpt in ('best', 'last'):
    model, ckpt_cfg = load_checkpoint(Path(f'{RUN}/{ckpt}.pt'), device, weights_dir=Path('weights'))
    model.eval()
    models[ckpt] = model
    for nms in NMS_RADII:
        t0 = time.time()
        cal = nbh.calibrate_tau(model, val_ds, device, TAUS, k=cfg.k, nms_radius=nms,
                                output_stride=cfg.output_stride,
                                match_radius_px=cfg.match_radius_px)
        calibrations[(ckpt, nms)] = cal
        chosen = cal['chosen']
        flag = '' if cal['tau_mae'] == cal['tau_f1'] else '   <- MAE and F1 argmins DISAGREE'
        print(f'{ckpt}.pt @NMS {nms}: tau(MAE-argmin) {cal["tau_mae"]:.2f} | '
              f'tau(F1-argmax) {cal["tau_f1"]:.2f} | at the chosen tau: '
              f'MAE {chosen["count_mae"]:.2f} RMSE {chosen["count_rmse"]:.2f} '
              f'bias {chosen["count_bias"]:+.2f} P {chosen["precision"]:.3f} '
              f'R {chosen["recall"]:.3f} F1 {chosen["f1"]:.3f} '
              f'({time.time() - t0:.0f}s){flag}')

In [ ]:
for ckpt in ('best', 'last'):
    path = nbh.plot_tau_sweeps(
        [(f'{ckpt}.pt @NMS {nms}', calibrations[(ckpt, nms)]) for nms in NMS_RADII],
        f'{RES}/figures/tau_sweep_{ckpt}.png',
        suptitle=(f'tau sweep, {ckpt}.pt on the {len(val_recs):,} Brackish val frames — '
                  'dashed = chosen (count-MAE argmin), dotted = F1 argmax where it differs'),
    )
    display(IPImage(str(path)))

# Selection records FIRST, then the raw sweeps: notebooks 3 and 4 read this file back with
# nbh.read_tau_calibration, which walks it for tau-shaped keys and takes the first match —
# and every sweep row carries its own "tau", so the chosen values must come first.
calibration_payload = {
    'selection': [
        {'checkpoint': ckpt, 'nms_radius': float(nms),
         'tau': calibrations[(ckpt, nms)]['chosen_tau'],
         'chosen_by': calibrations[(ckpt, nms)]['chosen_by'],
         'tau_f1_argmax': calibrations[(ckpt, nms)]['tau_f1'],
         'metrics': calibrations[(ckpt, nms)]['chosen']}
        for ckpt in ('best', 'last') for nms in NMS_RADII
    ],
    'taus': TAUS,
    'k': cfg.k,
    'match_radius_px': cfg.match_radius_px,
    'output_stride': cfg.output_stride,
    'run': RUN,
    'best_epoch': best_epoch,
    'n_val_images': len(val_recs),
    'sweeps': {f'{ckpt}|nms_{nms}': calibrations[(ckpt, nms)]['rows']
               for ckpt in ('best', 'last') for nms in NMS_RADII},
}
with open(f'{RES}/tau_calibration.json', 'w') as fh:
    json.dump(calibration_payload, fh, indent=2)
print('wrote', f'{RES}/tau_calibration.json')
print(json.dumps(calibration_payload['selection'], indent=1)[:1200])

## 9 · NMS spot checks — does radius 5 merge real neighbours?

The wheat report raised `nms_radius` from 1.5 to 5 because smeared wheat heads were decoding as duplicate points. Fish are not wheat heads: Brackish's stride-4 centre-cell collision rate is 0.0 %, but two fish *can* still swim within 5 output cells (20 px) of each other, and at that point a wider NMS is no longer removing duplicates — it is deleting animals.

The six **densest** val frames are the worst case for exactly that, so they are what gets looked at. Left column NMS 1.5, right column NMS 5, both decoded from `best.pt` at its calibrated τ; green = ground truth, red = predictions.

**Read it and decide.** If radius 5 loses a red ring that had a green dot under it, it is merging real neighbours and 1.5 is correct. If it only removes red rings that were stacked on one animal, it is doing its job. Table 2 below reports both either way; this figure is what makes the choice defensible rather than arbitrary.

In [ ]:
dense = nbh.densest_indices(val_recs, n=6)
print('densest val frames (index, GT count):',
      [(int(i), len(val_recs[i].points)) for i in dense])

spot_tau = calibrations[('best', 1.5)]['chosen_tau']
path = nbh.plot_nms_spot_checks(
    models['best'], val_ds, device, dense, f'{RES}/figures/nms_spot_checks.png',
    tau=spot_tau, k=cfg.k, output_stride=cfg.output_stride, radii=NMS_RADII,
)
display(IPImage(str(path)))

## 10 · Table 2 — best vs last

Same shape as the wheat report's Table 2, so the two are readable side by side. Each row is scored **at its own calibrated τ** (§ 8), which is the only fair way to compare decode settings; the τ values are in the footnote. Best in each column is bolded — lowest MAE/RMSE, lowest |bias|, highest P/R/F1.

**The argument for which checkpoint to report.** `best.pt` is selected by lowest *validation loss*, and validation loss here is dominated by the 56 % of val frames that contain no fish: on an empty frame the focal loss rewards a uniformly cold heatmap, and a model can keep driving that term down long after it has stopped getting better at finding fish. That is not hypothetical — **it is what the box run did**: its val loss bottomed at epoch 4 while every detection metric kept rising to epoch 8. If the same pattern shows in § 7's table (val loss bottoming early, F1 still climbing at epoch 8), then `last.pt` is the honest checkpoint to headline and `best.pt` is the one to report beside it, with the reason stated — not a quiet swap to whichever number is larger. If instead the val loss minimum coincides with the F1 maximum, `best.pt` stands on its own terms.

Either way **both are reported**, both are calibrated, and the choice is argued in the report rather than asserted here.

In [ ]:
rows = {}
for ckpt in ('best', 'last'):
    label = f'best epoch {best_epoch}' if ckpt == 'best' else f'last epoch {n_epochs}'
    for nms in NMS_RADII:
        cal = calibrations[(ckpt, nms)]
        rows[(ckpt, nms)] = {'model': label, 'config': f'NMS {nms:g}',
                             'tau': cal['chosen_tau'], 'metrics': cal['chosen']}
# Row order follows the wheat report (best first), with last @NMS 1.5 added: NMS 1.5 is this
# experiment's default, so the fourth row costs nothing and closes the 2x2.
entries = [rows[key] for key in (('best', 1.5), ('best', 5.0), ('last', 5.0), ('last', 1.5))]

table2_md = nbh.format_tuning_table(entries, k=cfg.k)
print(table2_md)

with open(f'{RES}/tuning_table.md', 'w') as fh:
    fh.write('# Table 2 — point head on CFD Brackish val '
             f'({len(val_recs):,} frames), decode settings\n\n')
    fh.write(table2_md)
    fh.write(f'\nRun `{RUN}` (seed 0, {n_epochs} epochs, 1 tile/frame); best epoch '
             f'{best_epoch} by lowest val loss. Every row is scored at its own '
             'calibrated tau (count-MAE argmin over 0.05-0.70).\n')
print('wrote', f'{RES}/tuning_table.md')
!ls -l {RES} {RES}/figures

## 11 · Next

`3_inference.ipynb` — predictions over the identical val frames, joined back to the box run through `cfd_id_map.json`; then `4_evaluate.ipynb` for the head-to-head. The artefacts this notebook leaves on Drive are everything those two need: `runs/brackish_points_s0/{best,last}.pt` + `history.json` + `train.log`, and `results/points/{tau_calibration.json, tuning_table.md, table1.md, figures/}`.

Nothing here is a claim yet. One seed, and a point path that saw roughly half the positive tiles per epoch that the box run's sampler fed it — both belong in the report beside any comparison.